# Initialize GEE python API

In [1]:
#import packages to run GEE python API

import ee #for google earth engine python api
import ipyleaflet #for GEE like interactive maps in python
import geemap  #for GEE like interactive maps in python
import pprint #for printing complex data structure with proper identation and line breaks
import time #for pausing, timing and timestamps


In [26]:

#import other packages for geospatial analysis

import os
import numpy as np
import pandas as pd
import os #for interacting with operating system
import re #regular expresssion -- for advanced string pattern matchin
import glob #for finding files using wildcards
import numpy as np #for arrays
import pandas as pd #for tabular data
import xarray as xr #for multi-dimensinal labeled arrays
import geopandas as gpd #for vector spatial data
import rioxarray #for connecting xarryy and GIS
import regionmask #for masking of gridded data by regions
import netCDF4 as nc
import rioxarray 
import warnings
from glob import glob

In [3]:
# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library
ee.Initialize(project='ee-diptibaral21') #this will produce token in you GEE account 
#which you need to put in the link generated here so you can initialize your GEE account here

# STEP 0: Setup and Counties

In [4]:
#defining study area

counties = ee.FeatureCollection("TIGER/2018/Counties")
selected_counties = ['Butte','Colusa','Glenn','Placer','Sacramento','San Joaquin',
                     'Sutter','Yolo','Yuba']

sac_valley = counties.filter(ee.Filter.eq('STATEFP', '06')) \
                     .filter(ee.Filter.inList('NAME', selected_counties))

#use "\" to indicate python that the line continues on the next line, otherwise it will 
#think the line ended there

In [5]:
#visualizing Sac Valley Counties

Map = geemap.Map()
Map.centerObject(sac_valley, 8)  

# Add the layer
Map.addLayer(sac_valley, {},'Sacramento Valley Counties')
Map

Map(center=[39.00285080659931, -121.59337946044376], controls=(WidgetControl(options=['position', 'transparent…

# STEP 1: Rice Masking

In [6]:
#function that masks rice from all the crops in the Crop Data Layer (CDL)
    #this is needed because we want to extract climate variables only from rice 
    #growing pixels within each county and not the whole county area
    #rice is coded as 3 in cropland category in the CDL

def mask_rice(image): 
   
    """
    Masks rice pixels from a Crop Data Layer (CDL) image.

    For a given CDL image:
    - Selects the 'cropland' band.
    - Identifies rice pixels (coded as 3 in CDL).
    - Masks the image to keep only rice pixels.
    - Clips the image to the Sacramento Valley region (`sac_valley`).
    - Sets non-rice pixels to 0 (binary: 1 = rice, 0 = non-rice).
    - Preserves the original image's 'system:time_start' property to maintain temporal metadata.

    Parameters:
    ----------
    image : ee.Image
        A single-year CDL image with a 'cropland' band.

    Returns:
    -------
    ee.Image
        A binary image where rice pixels are 1, non-rice pixels are 0, clipped to the Sacramento Valley.
    """
    
    rice_mask = image.select('cropland').eq(3)
    return image.updateMask(rice_mask) \
    .clip(sac_valley) \
    .unmask(0) \
    .set('system:time_start', image.get('system:time_start'))


#this function filters the cdl from start to end year and for each year image applies rice mask, 
#then creates rice sum which is the no of years rice grown from start to end year and then creates 
#rice frequency which is 1 if rice is grown was grown for at least one year  -- we are interested in 
#rice frequency because while defining the rice growing area  the area with atleast 1 year of cropping 
#is defined as rice growing area for this research

def get_rice_frequency(start_year, end_year):
   
    """
    Calculates rice presence and frequency over multiple years from the CDL.

    For the specified year range:
    - Filters the CDL ImageCollection to the given date range.
    - Selects the 'cropland' band from each image.
    - Applies `mask_rice` to keep only rice pixels for each year.
    - Reduces the collection by summing rice presence across years (`rice_sum`).
      This gives the total number of years rice was grown at each pixel.
    - Creates `rice_frequency`, a binary image where 1 indicates rice was grown 
      at least once during the period, and 0 otherwise.

    Parameters:
    ----------
    start_year : int
        The first year to include in the analysis.
    end_year : int
        The last year to include in the analysis.

    Returns:
    -------
    rice_sum : ee.Image
        An image where each pixel value represents the number of years rice was grown (range: 0 to number of years).
    rice_frequency : ee.Image
        A binary image where 1 indicates rice was grown at least once, 0 otherwise.
    """
    cdl = ee.ImageCollection("USDA/NASS/CDL") \
            .filterDate(f'{start_year}-01-01', f'{end_year}-12-31') \
            .map(lambda img: img.select(['cropland'])) \
            .map(mask_rice)

    rice_sum = cdl.reduce(ee.Reducer.sum())
    rice_frequency = rice_sum.gt(0)
    return rice_sum, rice_frequency
    
rice_sum, rice_frequency = get_rice_frequency(1997, 2023)   

Map.addLayer(rice_sum.clip(sac_valley),{
    'min':0, 
    'max': 27, 
    'palette': ['#e5f5e0', '#a1d99b', '#31a354']},'Rice Frequency (Years)')
Map
#palette from colorbrew

Map(bottom=25346.0, center=[39.00285080659931, -121.59337946044376], controls=(WidgetControl(options=['positio…

# STEP 2: Clipping LOCA2 Hybrid data to rice polygons

In [ ]:

# NOTE: run this file separately in HPC for faster processing

#file paths -- shared_dir to access data and my_dir to save data

CLIMATE_DIR = '/group/moniergrp/dbaral'
shared_dir = '/group/moniergrp/LOCA2_CA'
my_dir = os.path.join(CLIMATE_DIR, 'run_project/intermediate_data/loca_future_rice_nc')

#load rice polygons

shape_files = os.path.join(CLIMATE_DIR, 'run_project/input_data/shape_files')
rice = gpd.read_file(shape_files + '/Rice_Growing_Areas_30m.shp')


#loop through files in shared dir

for f in os.listdir(shared_dir):
    #only process if files that meet all below critera
    if (f.endswith('.nc') and
        ('tasmin' in f or 'tasmax' in f) and 
        ('ssp245' in f or 'ssp585' in f)):

        file_path = os.path.join(shared_dir, f)
        print(f"processing {f} ..")

        #open LOCA file
        ds = xr.open_dataset(file_path, engine = 'netcdf4', chunks={'time': 365})
        #select time slice 2020-2100
        ds = ds.sel(time=slice('2020-01-01', '2100-12-31'))
        clipped_vars={}
        #loop through all variables in the dataset
        for var in ds.data_vars:
            print(f' - Clipping variable: {var}')
            da = ds[var]
            da = da.rio.write_crs('EPSG:4326')
            clipped = da.rio.clip(rice.geometry, rice.crs)
            clipped_vars[var] = clipped
        #combine all clipped variables into one dataset
        clipped_ds = xr.Dataset(clipped_vars)
        #save output file
        out_file = os.path.join(my_dir, f.replace('.nc', '_rice.nc'))
        clipped_ds.to_netcdf(out_file)
        print(f'saved {out_file}\n')

print('Done all variables clipped and time-sliced for all files.') 

# STEP 3: Combining tmin and tmax LOCA Hybrid data files into one 

In [ ]:
# NOTE: run this file separately in HPC for faster processing

#file paths -- my_dir to access data and output_dir to save data
my_dir = '/group/moniergrp/dbaral/run_project/intermediate_data/loca_future_rice_nc'
output_dir = '/group/moniergrp/dbaral/run_project/intermediate_data/loca_future_rice_temp_nc'

models = ['ACCESS-CM2', 'CNRM-ESM2-1', 'EC-Earth3', 'EC-Earth3-Veg', 'GFDL-ESM4', 'INM-CM5-0', 'MPI-ESM1-2-HR', 'MRI-ESM2-0','FGOALS-g3', 'HadGEM3-GC31-LL', 'IPSL-CM6A-LR', 'KACE-1-0-G',  'MIROC6']
variables = ['tasmin', 'tasmax']
scenarios = ['ssp245', 'ssp585']


for model in models:
    for scenario in scenarios:
        datasets = []
        for var in variables:
            file_path = os.path.join(my_dir,f"{model}_{scenario}_r1i1p1f1_{var}_rice.nc") 
            if os.path.exists(file_path):
                ds = xr.open_dataset(file_path)
                datasets.append(ds)
            else:
                print(f"File not found: {file_path}")
        if datasets:
            combined_ds = xr.merge(datasets, compat = "override")
            #compute mean temperature
            combined_ds['tmean'] = (combined_ds['tasmin']+combined_ds['tasmax'])/2
            
            #growing season mask(May to october)
            mask = (
                ((combined_ds.time.dt.month >= 5) & (combined_ds.time.dt.month <=10)) 
            )
            # subset the data for all years
            subset = combined_ds.sel(time =mask)
        #save the combine dataset
        output_file = os.path.join(output_dir, f"{model}_{scenario}_r1i1p1f1_rice_temp.nc")
        subset.to_netcdf(output_file)
        print(f"Saved combined file: {output_file}")



# STEP 4: Processing loca combined data
### filtering from 2020-2100, converting k to C, calculating mean temps for each county

In [ ]:

# NOTE: run this file separately in HPC for faster processing

## Calculating the county level mean temperatures (min, max, and mean) 
    #netcdf files are daily records for each pixel,
    #we need to calculate mean temps(min, max, mean) by averging over all pixels in each county
    #we would need this mean value later to calculate gdd, and then based on that we define the growing stages
    #stress indices however is calculated pixel-wise first and then only aggregated over the county

#define file paths

CLIMATE_DIR = "/group/moniergrp/dbaral"

shape_file = os.path.join(CLIMATE_DIR, "run_project/input_data/shape_files")
loca_file = os.path.join(CLIMATE_DIR, "run_project/intermediate_data/loca_future_rice_temp_nc")
output_dir = os.path.join(CLIMATE_DIR, "run_project/intermediate_data/loca_future_rice_temp_csv")

#load shape file
counties = gpd.read_file(os.path.join(shape_file, "CA_9counties_shapefile.shp"))
#print(counties.columns)
#print(counties.head())

#process each netcdf file

all_files = [f for f in os.listdir(loca_file) if f.endswith(".nc")]
    #this creates a list of all netcdf files in the directory 

for file in all_files:
    print(f"Processing: {file}")
    #load loca file
    ds = xr.open_dataset(os.path.join(loca_file, file))
    ds = ds.rio.write_crs("EPSG:4326")
    
    combined_list = []
    #lopp through 9 counties
    for idx, row in counties.iterrows():
        county_name = row["NAME"]
        #spatially clip dataset using the county polygon
        masked = ds.rio.clip([row.geometry], counties.crs)
        #remove empty results 
        if masked.tasmin.count().item() == 0:
            print(f"No data for {county_name} in this file.")
            continue
        #mean over space - take the mean over spatial dimensions(lat and lon) for each day 
        df = (
            masked[['tasmin', 'tasmax', 'tmean']]
            .mean(dim=['lat', 'lon'])
            .to_dataframe()
        )

        #convert K to C
        df["tasmin"] = df["tasmin"] - 273.15
        df["tasmax"] = df["tasmax"] - 273.15
        df["tmean"] = df["tmean"] - 273.15
        #add county, year, month and day column
        df["county"] = county_name
        df["year"] = df.index.year
        df["month"] = df.index.month
        df["day"] = df.index.day
        #rename index to date
        df = df.rename_axis("date").reset_index()
        df["date"] = df["date"].dt.date
        #rename columns
        df = df.rename(columns ={
            "tasmin": "tmmn",
            "tasmax": "tmmx"})
        combined_list.append(df)
                            

    #combine all counties 
    final_df = pd.concat(combined_list)
    final_df.sort_index(inplace = True)

    #save one csv per file
    output_file = os.path.join(
        output_dir, 
        f"{file.replace('.nc', '')}_county_daily_means.csv"
    )
    final_df.to_csv(output_file)

    print(f"Saved: {output_file}")

# STEP 5: Calculating GDD 

In [7]:
#file paths

CLIMATE_DIR = "/group/moniergrp/dbaral"

file_path = os.path.join(CLIMATE_DIR, "run_project", "intermediate_data", "loca_future_rice_temp_csv")
save_path = os.path.join(CLIMATE_DIR, "run_project", "intermediate_data","loca_future_gdd")
os.makedirs(save_path, exist_ok=True)

#get all the files in filet path
all_files = [os.path.join(file_path, f) for f in os.listdir(file_path) if f.endswith(".csv")]

In [8]:
# base temp, lower temp, optimum temp and threshold gdd for each rice stage: needed to calculated daily gdd
#temp are in C
#referenced from Sharifi et al 2016 - for medium grain rice
# Pl_PI: Planting(PL) to Panicle Initiation(PI)
#PI_HD: Panicle Initiation (PI) to Heading (HD)
#HD_R7: Heading(HD) to R7 stage

growth_stages = {
    'PL_PI' : {
       'tbase': 10,
       'tlower': 14.30,
       'topt': 27.73,
       'threshold': 454
   },
   'PI_HD' : {
       'tbase': 9.8,
       'tlower': 15.37,
       'topt': 30.73,
       'threshold': 356
   },
   'HD_R7' : {
       'tbase': 8.9,
       'tlower': 15.93,
       'topt': 33.03,
       'threshold': 203
   }
}

In [9]:
#function to calculate daily gdd
def calculate_daily_gdd (tmmn, tmmx, tbase, tlower, topt):
  tmmx_adj = min(tmmx, topt)
  tmmn_adj = min(tmmn, tlower)
  tmean_adj = (tmmx_adj + tmmn_adj)/2
  gdd = max(tmean_adj - tbase, 0)
  return gdd

In [ ]:
#process each county and year combination

for file in all_files:  
    df = pd.read_csv(file)

    file_results = []
    
    for (county, year), group in df.groupby(['county', 'year']):
        season_data = group.copy()
        season_data = season_data.sort_values(by= 'date')

        #Convert 'date' column to datetime objects for reliable comparison
        season_data['date'] = pd.to_datetime(season_data['date'])

        #setting planting date to May 15 and harvest date to sep 15
        year_int = int(year)
        planting_date = pd.to_datetime(f"{year_int}-05-15")
        harvest_date  = pd.to_datetime(f"{year_int}-09-15")

        # Now you can filter within the season
        season_data = season_data[
            (season_data['date'] >= planting_date) &
            (season_data['date'] <= harvest_date)
        ]

        #Initialize growth tracking
        current_stage = 'PL_PI'
        stage_gdd = 0
        accumulated_gdd = 0
        transition_dates = {
            'pl_date': season_data.iloc[0]['date'] if not season_data.empty else None, # To handle empty season_data
            'pi_date': None,
            'hd_date': None,
            'r7_date': None,
            'grain_fill_start_date': None,
            'grain_fill_end_date': None
        }

        # Handling case where season_data might be empty after filtering
        if season_data.empty:
            print(f"No data for growing season in {county}, {year}")
            continue # Skip to the next group if no data for the season

        #daily process loop

        for index, row in season_data.iterrows():
            # Calculate daily GDD based on the current stage's parameters
            stage_params = growth_stages.get(current_stage)
            daily_gdd = 0

            if stage_params:
                daily_gdd = calculate_daily_gdd(
                    row['tmmn'],
                    row['tmmx'],
                    stage_params['tbase'],
                    stage_params['tlower'],
                    stage_params['topt']
                )
                stage_gdd += daily_gdd
                accumulated_gdd += daily_gdd


            # Check for stage transition based on GDD
            new_stage = None
            if current_stage in growth_stages:
                stage_params = growth_stages.get(current_stage)
                if stage_params and stage_gdd >= stage_params['threshold']:
                    if current_stage == 'PL_PI':
                        transition_dates['pi_date'] = row['date']
                        new_stage = 'PI_HD'
                    elif current_stage == 'PI_HD':
                        transition_dates['hd_date'] = row['date']
                        new_stage = 'HD_R7'
                    elif current_stage == 'HD_R7':
                        transition_dates['r7_date'] = row['date']
                        new_stage = 'completed' # Transition to a completed stage

            # Update current stage and reset stage_gdd if a transition occurred
            if new_stage:
                current_stage = new_stage
                stage_gdd = 0 # Reset stage_gdd for the new stage

            # Check for grain fill start date based on hd_date + 0 days
            if current_stage == 'completed' and transition_dates['hd_date'] is not None and transition_dates['grain_fill_start_date'] is None:
                transition_dates['grain_fill_start_date'] = transition_dates['hd_date']
                transition_dates['grain_fill_end_date'] = transition_dates['grain_fill_start_date'] + pd.Timedelta(days=30)


            # If in the grain fill period, update the stage
            if transition_dates['grain_fill_start_date'] is not None and transition_dates['grain_fill_end_date'] is not None:
                if row['date'] >= transition_dates['grain_fill_start_date'] and row['date'] <= transition_dates['grain_fill_end_date']:
                    current_stage = 'grain_fill'

            # save record
            if row['date'] <= harvest_date:
                # Add to overall results
                file_results.append({
                    'county': county,
                    'date': row['date'],
                    'year': year,
                    'month': row['month'],
                    'day': row['day'],
                    'stage': current_stage, # Use current_stage after transition check
                    'daily_gdd': daily_gdd,
                    'accumulated_gdd': accumulated_gdd,
                    'stage_gdd': stage_gdd, # stage_gdd is reset upon transition
                    'pl_date': transition_dates['pl_date'],
                    'pi_date': transition_dates['pi_date'],
                    'hd_date': transition_dates['hd_date'],
                    'r7_date': transition_dates['r7_date'],
                    'grain_fill_start_date': transition_dates['grain_fill_start_date'],
                    'grain_fill_end_date': transition_dates['grain_fill_end_date']
                })
    results_df = pd.DataFrame(file_results)
    
    #output_name = os.path.basename(file).replace(".csv","")+"_gdd_results.csv"
    #output_path = os.path.join(save_path, output_name)
    #results_df.to_csv(output_path, index = False)
    
    date_cols = ['pl_date', 'pi_date', 'hd_date', 'r7_date', 'grain_fill_start_date', 'grain_fill_end_date']
    results_df[date_cols] = results_df[date_cols].apply(pd.to_datetime)
    

    #calculating day of the year for each of the date cols
    for col in date_cols:
        results_df[f'{col}_doy'] = results_df[col].dt.dayofyear


    #converting date column to datetime format
    results_df['date'] = pd.to_datetime(results_df['date'])
    

    # Calculate days after planting for pi_date and hd_date
    results_df['pi_dap'] = (results_df['pi_date'] - results_df['pl_date']).dt.days
    results_df['hd_dap'] = (results_df['hd_date'] - results_df['pl_date']).dt.days
    results_df['r7_dap'] = (results_df['r7_date'] - results_df['pl_date']).dt.days
    results_df['grain_fill_dap'] = (results_df['grain_fill_start_date'] - results_df['pl_date']).dt.days
    
    output_name = os.path.basename(file).replace("rice_temp_county_daily_means.csv","")+"_rice_growth_stages_gdd_results_loca_future.csv"
    output_path = os.path.join(save_path, output_name)
    results_df.to_csv(output_path, index = False)
    print(f"Saved {output_name} {len(results_df)} rows") 

# STEP 6: Phenology Detection Using Growing Degree Day Model 

In [ ]:
#merge temp and gdd data to work for stress indices because we can identify the stages: Booting and Flowering only from gdd
#and to calulate stres indices we need daily temp data

# file paths

CLMATE_DIR = "/group/moniergrp/dbaral"
loca_temp_dir = os.path.join(CLIMATE_DIR, "run_project", "intermediate_data", "loca_future_rice_temp_csv")
loca_gdd_dir = os.path.join(CLIMATE_DIR, "run_project", "intermediate_data","loca_future_gdd")
save_dir = os.path.join(CLIMATE_DIR, "run_project", "intermediate_data", "loca_future_temp_gdd")
final_save_dir = os.path.join(CLIMATE_DIR, "run_project", "intermediate_data", "loca_future_growth_stages")

os.makedirs(save_dir, exist_ok=True)
os.makedirs(final_save_dir, exist_ok=True)

#get list of files in each dir

gdd_files = sorted([f for f in os.listdir(loca_gdd_dir) if f.endswith(".csv")])
temp_files = sorted([f for f in os.listdir(loca_temp_dir) if f.endswith(".csv")])


for gdd_file in gdd_files:
    base_name = gdd_file.split("_rice")[0]   
    temp_file = [f for f in temp_files if f.startswith(base_name)]

    if not temp_file:
        print(f" No match found for {gdd_file}")
        continue

    temp_file = temp_file[0]

    # read both
    df_gdd  = pd.read_csv(os.path.join(loca_gdd_dir, gdd_file))
    df_temp = pd.read_csv(os.path.join(loca_temp_dir, temp_file))

    # merge 
    merged = pd.merge(df_gdd, df_temp, on=["county","year","date", "month", "day"], how="outer")
    merged = merged.drop(columns = ["spatial_ref"])
    # save
    output_name = base_name + "_temp_gdd_merged.csv"
    merged.to_csv(os.path.join(save_dir, output_name), index=False)
    print(f" Merged: {output_name}")



In [ ]:
files = sorted([f for f in os.listdir(save_dir) if f.endswith(".csv")])

for file in files:
    print(f"processing for {file}")
    temp_gdd_df = pd.read_csv(os.path.join(save_dir, file))
                                                            
    results = []
    
    grouped = temp_gdd_df.groupby(['county', 'year'])
    
    #Loop through each group
    for i, ((county, year), group) in enumerate(grouped):
        group = group.copy() # Get the group DataFrame explicitly
        #print(f"Group {i}: {county}, {year}, {len(group)} rows")
        #print(group.dtypes)

        # convert dates to datetime format
        group['pi_date'] = pd.to_datetime(group['pi_date'], errors='coerce')
        group['hd_date'] = pd.to_datetime(group['hd_date'], errors='coerce')
        group['date']    = pd.to_datetime(group['date'], errors='coerce')
        #get first dates for this group
        first_pi_date = group['pi_date'].dropna().iloc[0]
        first_hd_date = group['hd_date'].dropna().iloc[0]
        #print(first_pi_date)
        #print(first_hd_date)
    
        booting_start = first_pi_date + pd.Timedelta(days = 7)
        booting_end = first_hd_date - pd.Timedelta(days = 7)
    
        flowering_start = first_hd_date - pd.Timedelta(days = 7)
        flowering_end = first_hd_date + pd.Timedelta(days = 7)
    
        grain_fill_start = first_hd_date
        grain_fill_end = first_hd_date + pd.Timedelta(days = 30)
    
        # Initialize the 'stage' column with a default value
    
        group['stage'] = 'growth stage' # Default stage
    
        #filtering for booting stage and naming the stage as booting
    
        group.loc[
            (group['date'] >= booting_start) &
            (group['date'] <= booting_end),
            'stage'
        ] = 'booting'
    
        #filtering for flowering stage and naming the stage column as flowering
        flowering_mask = (
            (group['date'] >= flowering_start) &
            (group['date'] <= first_hd_date)
        )
    
        group.loc[flowering_mask, 'stage'] = 'flowering'
    
        #filtering for overlapping period of flowering and grain fill stage and naming it flowering and grainfill
    
        flowering_grainfill_mask = (
            (group["date"] > first_hd_date) &
            (group["date"] <= flowering_end)
        )
    
        group.loc[flowering_grainfill_mask, 'stage'] = 'flowering_grainfill'
    
    
        #filtering for grain_fil period and naming it as grainfill
    
        grain_fill_mask = (
            (group['date'] > flowering_end) &
            (group['date'] <= grain_fill_end)
        )
    
        group.loc[grain_fill_mask, 'stage'] = 'grain_fill'

    
        # Add tmmx_gf and tmmn_gf columns for grain fill period
        group['tmmx_gf'] = np.nan
        group['tmmn_gf'] = np.nan
    
        # Calculate max and min temperature for the entire grain fill period for this county and year
        grain_fill_period_data = group[group['stage'] == 'grain_fill']
        if not grain_fill_period_data.empty:
            max_tmmx_gf = grain_fill_period_data['tmmx'].max()
            min_tmmn_gf = grain_fill_period_data['tmmn'].min()
            # Assign the calculated max and min to all rows within the grain fill period for this group
            group.loc[group['stage'] == 'grain_fill', 'tmmx_gf'] = max_tmmx_gf
            group.loc[group['stage'] == 'grain_fill', 'tmmn_gf'] = min_tmmn_gf
    
        results.append(group)
    
    #combine all groups
    final_results = pd.concat(results)
    output_name = file.replace(".csv", "_final_growth_stages.csv")
    final_results.to_csv(os.path.join(final_save_dir, output_name), index = False)
    print(f"Saved {output_name}")

# STEP 7: Stage Table (Python)

In [ ]:
#define file paths

CLIMATE_DIR = "/group/moniergrp/dbaral"

file_path = os.path.join(CLIMATE_DIR, "run_project", "intermediate_data", "loca_future_growth_stages")
save_path = os.path.join(CLIMATE_DIR, "run_project", "intermediate_data", "loca_future_stage_table")

# make sure output dir exists
os.makedirs(save_path, exist_ok=True)

all_files = [os.path.join(file_path, f) for f in os.listdir(file_path) if f.endswith("final_growth_stages.csv")]

#In step 4 we got the growth stage column for each row of data, now we need a table that just gives stage, start date and end date
#this table is needed to calculate the stress indices

# Define the stages
stage_map = {
    "booting":    ["booting"],
    "flowering":  ["flowering", "flowering_grainfill"],   # include overlap
    "grain_fill": ["grain_fill", "flowering_grainfill"],  # include overlap
}

for file in all_files:
    df = pd.read_csv(file)

    # ensure date is datetime
    df["date"] = pd.to_datetime(df["date"])

    rows = []
    for (county, year), g in df.groupby(["county", "year"], sort=True):
        for out_stage, labels in stage_map.items():
            sub = g[g["stage"].isin(labels)]
            if sub.empty:
                continue
            rows.append({
                "county": county,
                "year": int(year),
                "stage": out_stage,
                "start_date": sub["date"].min().date().isoformat(),
                "end_date": sub["date"].max().date().isoformat(),
            })

    stage_dates = (pd.DataFrame(rows)
                     .sort_values(["county", "year", "stage"])
                     .reset_index(drop=True))

    output_name = os.path.basename(file).replace("_temp_gdd_merged_final_growth_stages.csv", "") + "_stage_table.csv"
    out_csv = os.path.join(save_path, output_name)

    stage_dates.to_csv(out_csv, index=False)

    print("Saved:", out_csv)
    print(stage_dates.head(10))

# STEP 8 a: Stress Calculation Functions

In [ ]:
# GEE-equivalent county/year/stage stress indices from LOCA2 NetCDFs
# - Reads 20 NetCDF files (10 models × 2 SSPs)
# - Vars: tmmn, tmmx, tmean (dims: time, lat, lon)
# - For each county × year × stage window:
#     * daily stress per pixel (booting/flowering logic)
#     * stage-window climate means (tmmn/tmmx/tmean)
#     * stage-window stress sums (cdstress_bo/cdstress_fl/htstress_fl)
#     * county reduction = AREA-WEIGHTED mean using cos(lat) weights (like equal-area intent in GEE)
# - Exports one CSV per NetCDF file

In [17]:

#here we will calculate stress indices based on the above formula for each pixel -- then aggregate those pixel values over each 
#county to get the county mean stress for each year

#defining the thresholds

booting_cold = 13
flowering_cold = 10.9
flowering_heat = 37.5


def calculate_daily_stress(ds: xr.Dataset, stage: str) -> xr.Dataset:

    """
    Calculate daily temperature stress indices for rice based on phenological stage.

    This function computes pixel-level daily cold and/or heat stress using
    stage-specific temperature thresholds from Espe et al. (2017). Stress is
    calculated for *all dates*, but only the indices relevant to the specified
    growth stage are populated; other stress variables remain zero and can be
    masked later using stage-specific calendars.

    Cold stress is defined as the positive difference between a critical
    temperature threshold and daily minimum temperature. Heat stress is defined
    as the positive difference between daily maximum temperature and a critical
    heat threshold.

    Parameters
    ----------
    ds : xr.Dataset
        Xarray Dataset containing daily temperature variables with dimensions
        (time, lat, lon) or equivalent. Must include:
        - tasmin : daily minimum temperature (°C)
        - tasmax : daily maximum temperature (°C)
        - tmean  : daily mean temperature (°C)

    stage : str
        Rice phenological stage for which stress should be computed.
        Supported values are:
        - "booting"   : cold stress during booting stage
        - "flowering" : cold and heat stress during flowering stage
        - "grain_fill": no temperature stress applied (all stress indices = 0)

    Returns
    -------
    xr.Dataset
        Dataset containing the original temperature variables and the following
        stress indices (all aligned to the input dimensions and coordinates):

        - cdstress_bo : Cold stress during booting stage (°C·day)
        - cdstress_fl : Cold stress during flowering stage (°C·day)
        - htstress_fl : Heat stress during flowering stage (°C·day)

        Stress variables not relevant to the selected stage are returned as
        zero-filled arrays to ensure consistent structure across stages.

    Notes
    -----
    - Thresholds are fixed and based on Espe et al. (2017):
        * Booting cold threshold: 13.0 °C
        * Flowering cold threshold: 10.9 °C
        * Flowering heat threshold: 37.5 °C
    - Stress indices are computed per pixel and per day and are intended to be
      aggregated temporally and spatially (e.g., county means) in downstream
      analysis.
    - xr.zeros_like is used to preserve metadata, coordinates, and dimensions.

    Raises
    ------
    ValueError
        If `stage` is not one of the supported growth stages.
    """    
        
    #for loca data tmin and tmax are named as tasmin and tasmax
    tmmn = ds["tasmin"]  
    tmmx = ds["tasmax"]
    tmean = ds["tmean"]


    # ALWAYS define these so they're never missing
    #xr.zero_like(other_array) -- it creats a zero-filled array that copies the shape, 
    #dimensions, and coords of the existing 
    #DataArray or Dataset -- np.zeros(da.shape) looses metadata no dims, no coords, 
    #no labels so xr.zerolike()
    #first we create dataset for stress indices similar to tmmn, tmmx with all 0, 
    #then later add the stress indices values
    #specific to stages define by if statements below
    
    cdstress_bo = xr.zeros_like(tmmn).rename("cdstress_bo")
    cdstress_fl = xr.zeros_like(tmmn).rename("cdstress_fl")
    htstress_fl = xr.zeros_like(tmmx).rename("htstress_fl")

    if stage == "booting":
        cdstress_bo = xr.where(
            tmmn < booting_cold,
            (booting_cold - tmmn),
            0.0
        ).rename("cdstress_bo")
        
        #renaning is important here because we are using tmmn band here so xarrya takes 
        #the name form tmmn - but it is no longer temp-its cold stress, although we name 
        #the variable cold stress, we need to give the new band cold stress name

    elif stage == "flowering":
        cdstress_fl = xr.where(
            tmmn < flowering_cold,
            (flowering_cold - tmmn),
            0.0
        ).rename("cdstress_fl")

        htstress_fl = xr.where(
            tmmx > flowering_heat,
            (tmmx - flowering_heat),
            0.0
        ).rename("htstress_fl")

    elif stage == "grain_fill":
        # keep zeros 
        pass

    else:
        raise ValueError(f"Unknown stage: {stage}")

    return xr.Dataset(
        {
            "tmmn": tmmn,
            "tmmx": tmmx,
            "tmean": tmean,
            "cdstress_bo": cdstress_bo,
            "cdstress_fl": cdstress_fl,
            "htstress_fl": htstress_fl,   
        }
    )


In [18]:
#County masks + area weights (lat/lon)
    #convert counties into a maskable region object
    #this function is to build a mask using regionmask.Regions which assign each grid cell in a raset (here LOCA) 
    #to a region(county) and creats masks like which pixels belong to which county
    #this is useful for aggregating LOCA data by county -- in GEE reduce by regions is used

def build_county_mask(counties_gdf: gpd.GeoDataFrame, name_col: str = "NAME"):
    """
    Create a regionmask Regions object from a counties GeoDataFrame.

    This function reprojects the input GeoDataFrame to WGS84 (EPSG:4326)
    and constructs a `regionmask.Regions` object using each county's
    geometry as a separate region. County names are used both as region
    names and abbreviations.

    Parameters
    ----------
    counties_gdf : geopandas.GeoDataFrame
        GeoDataFrame containing county polygon geometries.
    name_col : str, default "NAME"
        Column in `counties_gdf` that contains county names to label
        each region.

    Returns
    -------
    regionmask.Regions
        A reusable Regions object that can be applied to gridded data
        (e.g., xarray datasets) to generate county-level masks.
    """
    gdf = counties_gdf.to_crs("EPSG:4326").copy()

    regions = regionmask.Regions(
        outlines=list(gdf.geometry),  #list of polygon shapes (one per county)
        names=list(gdf[name_col].astype(str).values), #name for each county
        abbrevs=list(gdf[name_col].astype(str).values), #abbreviation used internally by regionmask here they are same as names but sometimes people use codes like FIPS
        name="counties", #labels the whole collection as counties
    )
    return regions #returns reuseable object

In [19]:

#this fuction calculates county-level yearly stress indices for specific rice growth stage
# this is the Python/xarry equivalent of Google Earth Engine reducer pipeline:
#EE image collection -- time filter -- per-pixel stress - reduce over time-- reduce over region

def calculate_yearly_stage_stress(stage_feature,
                                  growing_season_images: xr.Dataset,
                                  regions: regionmask.Regions,
                                  mask: xr.DataArray,
                                  weights: xr.DataArray,
                                  name_col: str = "NAME"):
    """
    Python equivalent of EE pipeline.

    Parameters
    ----------
    stage_feature : dict-like or pd.Series
        Must have keys: 'county', 'year', 'stage', 'start_date', 'end_date'
    growing_season_images : xr.Dataset
        Must have vars: 'tmmn','tmmx' (optional 'tmean') and coords 'time'
        Spatial coords should match what mask/weights were built on.
    regions : regionmask.Regions
        Regions created from county polygons (EPSG:4326) using build_county_mask.
    mask : xr.DataArray
        County id mask on the dataset grid (e.g., regions.mask(ds['lon'], ds['lat']))
    weights : xr.DataArray
        Area weights on the grid (e.g., cos(lat) for lat/lon grid), broadcastable to (lat,lon)

    Returns
    -------
    pd.DataFrame
        One-row dataframe with:
        county, stage, year, tmmn, tmmx, tmean, cdstress_bo, cdstress_fl, htstress_fl
        where climate vars are time-means and stress vars are time-sums,
        all spatially averaged over the county polygon.
    """

    #parse stage feature
    county = str(stage_feature["county"])
    year   = int(stage_feature["year"])
    stage  = str(stage_feature["stage"])
    start  = pd.to_datetime(stage_feature["start_date"])
    end    = pd.to_datetime(stage_feature["end_date"])

    # county id in the regionmask object 
        # Map the county name to its corresponding numeric region ID in the RegionMask object.
        # This is needed because spatial data arrays use numeric IDs to identify regions,
        # not names. We first create a dictionary mapping names to IDs, check if the
        # requested county exists, and then retrieve its ID (rid) for use in masking
        # or extracting data specific to that county.
    
    name_to_id = {n: i for i, n in enumerate(regions.names)}
    if county not in name_to_id:
        raise ValueError(f"County '{county}' not found in regions.names.")
    rid = name_to_id[county]

    #subset dataset by stage window 
    ds_sub = growing_season_images.sel(time=slice(start, end))

    # daily stress  
    daily_stress = calculate_daily_stress(ds_sub, stage)  

    # time aggregation (matches EE)
    climate_means = daily_stress[["tmmn", "tmmx", "tmean"]].mean("time", skipna=True)
    stress_sums   = daily_stress[["cdstress_bo", "cdstress_fl", "htstress_fl"]].sum("time", skipna=True)

    yearly = xr.merge([climate_means, stress_sums])

    # spatial reduction by county (mean over pixels inside county) 
        # Compute the county-level average for each variable:
        # 1. Mask pixels outside the target county using mask == rid, so only pixels
        #    inside the county contribute to the calculation.
        # 2. Use a weighted mean across spatial dimensions (lat/lon) to account for
        #    differences in pixel area or grid distortion (e.g., higher latitude pixels
        #    cover smaller physical area). This ensures the county average reflects
        #    the true area-weighted value, not just a simple pixel mean.
        # 3. skipna=True ignores missing or masked pixels outside the county.

    out = {}
    for var in yearly.data_vars:
        da = yearly[var].where(mask == rid)
        # weights expected to be compatible with da's spatial dims ( lat/lon)
        out[var] = float(da.weighted(weights).mean(dim=[d for d in da.dims if d != "time"], skipna=True).values)

    # output
    out_row = {
        "county": county,
        "stage": stage,
        "year": year,
        "tmmn": out.get("tmmn", np.nan),
        "tmmx": out.get("tmmx", np.nan),
        "tmean": out.get("tmean", np.nan),
        "cdstress_bo": out.get("cdstress_bo", np.nan),
        "cdstress_fl": out.get("cdstress_fl", np.nan),
        "htstress_fl": out.get("htstress_fl", np.nan),
    }

    return pd.DataFrame([out_row])


# Step 8 b: Preparing Stage Table with stress indices

In [24]:
CLIMATE_DIR = "/group/moniergrp/dbaral"

# paths

stage_table_dir = os.path.join(CLIMATE_DIR, "run_project/intermediate_data/loca_future_stage_table")
out_dir = os.path.join(CLIMATE_DIR, "run_project/intermediate_data/loca_future_stage_stress_tables")
loca_netcdf_dir = os.path.join(CLIMATE_DIR, "run_project/intermediate_data/loca_future_rice_temp_nc")
shapefile = os.path.join(CLIMATE_DIR, "run_project/input_data/shape_files/CA_9counties_shapefile.shp")

# make sure output dir exists
os.makedirs(out_dir, exist_ok=True)

# counties -- regions
counties_gdf = gpd.read_file(shapefile).to_crs("EPSG:4326")
regions = build_county_mask(counties_gdf)


# Helper function top load matching LOCA dataset


def load_loca_dataset(stage_table_file, netcdf_dir):
    """
    Matches stage table:
      ACCESS-CM2_ssp245_r1i1p1f1__stage_table.csv
    to NetCDF:
      ACCESS-CM2_ssp245_r1i1p1f1_rice_temp.nc
    """
    base = os.path.basename(stage_table_file)
    model = base.split("_ssp")[0]

    m = re.search(r"(ssp\d{3})", base)
    if not m:
        raise ValueError(f"Could not parse SSP from {base}")
    ssp = m.group(1)

    nc_path = os.path.join(netcdf_dir, f"{model}_{ssp}_r1i1p1f1_rice_temp.nc")
    if not os.path.exists(nc_path):
        raise FileNotFoundError(f"Missing NetCDF:\n{nc_path}")

    print(f"[load_loca_dataset] Using: {nc_path}")
    ds = xr.open_dataset(nc_path)

    # converting Kelvin to Celsius
        #in loca netcdf file, temps are in K so change to C
    u = str(ds["tasmin"].attrs.get("units", "")).lower()
        # .attrs to get the metadata, .get to access the valye for the key units (its in dictionary form )
        # usging .attrs['units'] won't work because the metadata is in dict form
    vmin = float(ds["tasmin"].min())
    if ("k" in u) or (vmin > 100):
        ds = ds.copy()
        ds["tasmin"] = ds["tasmin"] - 273.15
        ds["tasmax"] = ds["tasmax"] - 273.15
        ds["tasmin"].attrs["units"] = "degC"
        ds["tasmax"].attrs["units"] = "degC"

    # renaming the coords names for regionmask
    rename = {}
    if "longitude" in ds.coords and "lon" not in ds.coords:
        rename["longitude"] = "lon"
    if "latitude" in ds.coords and "lat" not in ds.coords:
        rename["latitude"] = "lat"
    if rename:
        ds = ds.rename(rename)

    return ds

    ds = kelvin_to_celsius(ds)

# MAIN LOOP (20 tables)

stage_tables = sorted(glob.glob(os.path.join(stage_table_dir, "*_stage_table.csv")))
    #this gives the stage tables paths

for st_path in stage_tables:
    tag = os.path.basename(st_path).replace("_stage_table.csv", "")
    print(f"\nProcessing: {tag}")

    # load stage table
    stage_table_df = pd.read_csv(st_path)
    stage_table_df["start_date"] = pd.to_datetime(stage_table_df["start_date"])
    stage_table_df["end_date"]   = pd.to_datetime(stage_table_df["end_date"])
    stage_table_df["year"]       = stage_table_df["year"].astype(int)

    # load matching LOCA data
    ds = load_loca_dataset(st_path, loca_netcdf_dir)

    # build mask + weights for THIS grid
        # Build a region mask that assigns each grid cell to a county,
        # and latitude-based area weights (cos(lat)) to account for decreasing
        # grid-cell area toward higher latitudes when computing spatial averages
    
    mask = regions.mask(ds["lon"], ds["lat"])          # (lat, lon) -- always remember x is lon and y is lat
    weights = np.cos(np.deg2rad(ds["lat"]))             # latitude weights

    # This step is equivalent to stage_table.map(...).flatten() in GEE

    #Initialize an empty list to store results
    rows = []
    #loop over each row of the df - each row represents a stage feature for a specific county and year
    for idx, row in stage_table_df.iterrows():
        #convert the row to a dictionary and calculate stress indices for that row

        out = calculate_yearly_stage_stress(
            stage_feature=row.to_dict(), # the county/year/stage info
            growing_season_images=ds, # climate data (xr.Dataset) for the growing season
            regions=regions, #county polygons 
            mask=mask, # mask to select only rice pixels
            weights=weights #pixel area weights
        )
        rows.append(out)

    if len(rows) == 0:
        print(f"No stress calculated for {tag}")
        all_year_table = pd.DataFrame(columns=["county", "year", "stage", "cdstress_bo", "cdstress_fl", "htstress_fl"])
    else:
        all_year_table = pd.concat(rows, ignore_index=True)
        #if its an empty row, it will give empty df with heading as defined, if not then it will be concatenated to the rows[]. 
        #its good idea to create the empty df to check if the rows are empty or not

    # selecte only the columns needed
    all_year_table_selected = all_year_table[
        ["county", "year", "stage",
         "cdstress_bo", "cdstress_fl", "htstress_fl"]
    ]

    # checking the first feature 
    first_export_feature = all_year_table_selected.iloc[0].to_dict()
    print("First feature:", first_export_feature)

    # save
    out_csv = os.path.join(out_dir, f"{tag}_yearly_stage_stress.csv")
    all_year_table_selected.to_csv(out_csv, index=False)
    print(f"Saved  {out_csv}")




Processing: ACCESS-CM2_ssp245_r1i1p1f1_
[load_loca_dataset] Using: /group/moniergrp/dbaral/run_project/intermediate_data/loca_future_rice_temp_nc/ACCESS-CM2_ssp245_r1i1p1f1_rice_temp.nc
First feature: {'county': 'Butte', 'year': 2020, 'stage': 'booting', 'cdstress_bo': 0.0, 'cdstress_fl': 0.0, 'htstress_fl': 0.0}
Saved  /group/moniergrp/dbaral/run_project/intermediate_data/loca_future_stage_stress_tables/ACCESS-CM2_ssp245_r1i1p1f1__yearly_stage_stress.csv

Processing: ACCESS-CM2_ssp585_r1i1p1f1_
[load_loca_dataset] Using: /group/moniergrp/dbaral/run_project/intermediate_data/loca_future_rice_temp_nc/ACCESS-CM2_ssp585_r1i1p1f1_rice_temp.nc
First feature: {'county': 'Butte', 'year': 2020, 'stage': 'booting', 'cdstress_bo': 0.12774993479251862, 'cdstress_fl': 0.0, 'htstress_fl': 0.0}
Saved  /group/moniergrp/dbaral/run_project/intermediate_data/loca_future_stage_stress_tables/ACCESS-CM2_ssp585_r1i1p1f1__yearly_stage_stress.csv

Processing: CNRM-ESM2-1_ssp245_r1i1p1f1_
[load_loca_dataset] 

# STEP 9: Model input data preparation

In [30]:
#import stage file and climate file

#file paths
CLIMATE_DIR = "/group/moniergrp/dbaral"
file_path = os.path.join(CLIMATE_DIR, "run_project", "intermediate_data")
save_dir =  os.path.join(CLIMATE_DIR, "run_project", "input_data", "loca_future_model_input")

# make sure output dir exists
os.makedirs(save_dir, exist_ok=True)

#get all the files
climate_files = sorted(glob(file_path + "/loca_future_rice_temp_csv/*.csv"))
stage_files = sorted(glob(file_path + "/loca_future_stage_table/*.csv"))
stress_files = sorted(glob(file_path + "/loca_future_stage_stress_tables/*.csv"))

final_dfs = []

#loop over all files

for i in range(len(climate_files)):
    climate_fp = climate_files[i]
    climate = pd.read_csv(climate_files[i], parse_dates = ["date"])
    stages = pd.read_csv(stage_files[i], parse_dates = ["start_date", "end_date"])
    stress = pd.read_csv(stress_files[i])
    
    #merge climate and stage
    merged = climate.merge(stages, on = ["county", "year"], how = "left")
    merged = merged[(merged["date"] >= merged["start_date"]) & (merged["date"] <= merged["end_date"])]
    
    #split stages
    
    # suffix dictionary for stages
    stage_suffix = {
        "booting": "bo",
        "flowering": "fl",
        "grain_fill": "gf"
    }
    
    # function to split stage-specific data and rename columns
    def split_and_rename(df, stage, suffix):
        out = df[df["stage"] == stage].copy()
        out = out.rename(columns={
            "tmmn": f"tmmn_{suffix}",
            "tmmx": f"tmmx_{suffix}",
            "tmean": f"tmean_{suffix}"
        })
        return out
    
    # function to aggregate annual mean
    def aggregate_annual_mean(df, suffix):
        return df.groupby(["county", "year"], as_index=False).agg({
            f"tmmn_{suffix}": "mean",
            f"tmmx_{suffix}": "mean",
            f"tmean_{suffix}": "mean"
        })
        
    booting_df = split_and_rename(merged, "booting", "bo")
    flowering_df = split_and_rename(merged, "flowering", "fl")
    grainfill_df = split_and_rename(merged, "grain_fill", "gf")               

    # seasonal data May 11 to Oct 6
    seasonal_df = climate[
        ((climate["date"].dt.month > 5) | ((climate["date"].dt.month == 5) & (climate["date"].dt.day >= 11))) &
        ((climate["date"].dt.month < 10) | ((climate["date"].dt.month == 10) & (climate["date"].dt.day <= 6)))
    ]
    
    booting_yr = aggregate_annual_mean(booting_df, "bo")
    flowering_yr = aggregate_annual_mean(flowering_df, "fl")
    grainfill_yr = aggregate_annual_mean(grainfill_df, "gf")

    seasonal_yr = seasonal_df.groupby(["county", "year"], as_index=False).agg({
        "tmmn": "mean",
        "tmmx": "mean",
        "tmean": "mean"
    })
    
    # rename seasonal columns to *_gs
    seasonal_yr = seasonal_yr.rename(columns={
        'tmmn': 'tmmn_gs',
        'tmmx': 'tmmx_gs',
        'tmean': 'tmean_gs'
    })
    # stress yearly aggregation
    stress_yr = stress.groupby(["county", "year"], as_index=False).agg({
        "cdstress_bo": "max",
        "cdstress_fl": "max",
        "htstress_fl": "max"
    })
    
    # merge all together
    final_df = (
        booting_yr
        .merge(flowering_yr, on=["county", "year"], how="inner")
        .merge(grainfill_yr, on=["county", "year"], how="inner")
        .merge(seasonal_yr, on=["county", "year"], how="inner")
        .merge(stress_yr, on=["county", "year"], how="inner")
    )
    
    # save files
    base = os.path.basename(climate_fp)
    model_name = base.split("_rice")[0]
    output_name = f"{model_name}_Lasso_Model_Input_2020_2100.csv"
    save_path = os.path.join(save_dir, output_name)
    final_df.to_csv(save_path, index = False)
